In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, func, cast, String
from sqlalchemy.dialects.sqlite import insert as sqlite_insert
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.config import db_engine
from app.models import (
    SilverCleanAd,
    DimProduct
)

In [ ]:
with db_engine.connect() as connection:
    df_silver_products = pl.read_database(
        select(
            SilverCleanAd.baseline_id,
            SilverCleanAd.category,
            SilverCleanAd.brand,
            SilverCleanAd.cpu_brand,
            SilverCleanAd.cpu_model,
            SilverCleanAd.ram_gb,
            SilverCleanAd.storage_gb,
            func.concat_ws(
                ', ', 
                func.nullif(SilverCleanAd.cpu_model, 'Brand Not Informed'),
                cast(func.nullif(SilverCleanAd.ram_gb, 0), String) + 'GB RAM',
                cast(func.nullif(SilverCleanAd.storage_gb, 0), String) + 'GB SSD'
            ).label('specs_summary')
        )
        .where(SilverCleanAd.baseline_id.is_not(None)),
        connection=connection
    )

In [ ]:
df_dim_products = (
    df_silver_products
    .unique(subset=['baseline_id'], keep='first')
)

In [ ]:
if not df_dim_products.is_empty():
    with Session(db_engine) as session:
        stmt = sqlite_insert(DimProduct).values(df_dim_products.to_dicts())
        stmt = stmt.on_conflict_do_nothing(index_elements=['baseline_id'])
        session.execute(stmt)
        session.commit()
else:
    print("No data found to insert.")